In [2]:
# ============================================================
# CS-340 Project Two - Grazioso Salvare Dashboard
# ============================================================

# ------------------------------------------------------------
# Setup JupyterDash
# ------------------------------------------------------------
from jupyter_dash import JupyterDash

# Dash components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output

# Other libraries
import base64
import pandas as pd

JupyterDash.infer_jupyter_proxy_config()


# ------------------------------------------------------------
# CRUD Python Module
# ------------------------------------------------------------

# Change these if your CRUD file/class has a different name.
from CRUD_Python_Module import AnimalShelter


# ------------------------------------------------------------
# Database Connection
# ------------------------------------------------------------

username = "aacuser"
password = "SNHU123!"

db = AnimalShelter(username, password)


# ------------------------------------------------------------
# Load Initial Dataset
# ------------------------------------------------------------

# Empty query retrieves all records.
df = pd.DataFrame.from_records(db.read({}))

# Remove MongoDB ObjectID because Dash DataTable cannot display it.
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


# ------------------------------------------------------------
# Grazioso Salvare Logo
# ------------------------------------------------------------

# Grazioso Salvare Logo.
image_filename = 'Grazioso Salvare Logo.png'

encoded_image = base64.b64encode(
    open(image_filename, 'rb').read()
).decode()


# ------------------------------------------------------------
# Create Dashboard
# ------------------------------------------------------------

app = JupyterDash(__name__)


# ------------------------------------------------------------
# Dashboard Layout
# ------------------------------------------------------------

app.layout = html.Div(

    [

        # ====================================================
        # HEADER / BRANDING
        # ====================================================

        html.Div(
            [

                html.A(
                    html.Img(
                        src='data:image/png;base64,{}'.format(encoded_image),
                        style={
                            'width': '240px',
                            'height': 'auto',
                            'display': 'block',
                            'margin': '0 auto'
                        }
                    ),
                    href='https://www.snhu.edu',
                    target='_blank'
                ),

                html.H1(
                    'Grazioso Salvare Rescue Animal Dashboard',
                    style={
                        'fontSize': '28px',
                        'margin': '5px 0 0 0',
                        'textAlign': 'center'
                    }
                ),

                html.H4(
                    'Created by Matthew Owens',
                    style={
                        'fontSize': '15px',
                        'margin': '3px 0 10px 0',
                        'textAlign': 'center'
                    }
                )

            ],

            style={
                'textAlign': 'center',
                'padding': '10px 0 5px 0'
            }
        ),

        html.Hr(
            style={
                'margin': '5px 0 10px 0'
            }
        ),

        # ====================================================
        # INTERACTIVE FILTER OPTIONS
        # ====================================================

        html.Div(
            [

                html.H3(
                    'Rescue Type Filter',
                    style={
                        'fontSize': '20px',
                        'margin': '5px 0 10px 0',
                        'textAlign': 'center'
                    }
                ),

                dcc.RadioItems(
                    id='filter-type',

                    options=[
                        {
                            'label': ' Water Rescue',
                            'value': 'water'
                        },
                        {
                            'label': ' Mountain or Wilderness Rescue',
                            'value': 'mountain'
                        },
                        {
                            'label': ' Disaster or Individual Tracking',
                            'value': 'disaster'
                        },
                        {
                            'label': ' Reset',
                            'value': 'reset'
                        }
                    ],

                    value='reset',

                    labelStyle={
                        'display': 'inline-block',
                        'marginRight': '25px',
                        'fontSize': '15px'
                    },

                    style={
                        'textAlign': 'center'
                    }
                )

            ],

            style={
                'padding': '5px 0 10px 0'
            }
        ),

        html.Hr(
            style={
                'margin': '5px 0 10px 0'
            }
        ),

        # ====================================================
        # INTERACTIVE DATA TABLE
        # ====================================================

        dash_table.DataTable(

            id='datatable-id',

            columns=[
                {
                    "name": i,
                    "id": i,
                    "deletable": False,
                    "selectable": True
                }
                for i in df.columns
            ],

            data=df.to_dict('records'),

            page_size=10,

            sort_action='native',

            filter_action='native',

            row_selectable='single',

            selected_rows=[0],

            style_table={
                'overflowX': 'auto',
                'width': '100%'
            },

            style_cell={
                'textAlign': 'left',
                'padding': '6px',
                'fontSize': '12px',
                'minWidth': '90px',
                'maxWidth': '220px',
                'whiteSpace': 'normal'
            },

            style_header={
                'fontWeight': 'bold',
                'fontSize': '12px',
                'padding': '7px'
            },

            style_data={
                'height': '32px'
            }

        ),

        html.Br(),

        # ====================================================
        # CHARTS
        # ====================================================

        html.Div(
            [

                # Pie Chart
                html.Div(
                    id='graph-id',
                    style={
                        'width': '50%',
                        'padding': '5px'
                    }
                ),

                # Geolocation Map
                html.Div(
                    id='map-id',
                    style={
                        'width': '50%',
                        'padding': '5px'
                    }
                )

            ],

            style={
                'display': 'flex',
                'width': '100%',
                'alignItems': 'stretch'
            }
        )

    ],

    style={
        'width': '98%',
        'margin': '0 auto',
        'fontFamily': 'Arial, sans-serif'
    }

)


# ============================================================
# Controller - Interactive Filtering
# ============================================================

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    # --------------------------------------------------------
    # Reset
    # --------------------------------------------------------

    if filter_type == 'reset':

        query = {}


    # --------------------------------------------------------
    # Water Rescue
    #
    # Breeds:
    # Labrador Retriever Mix
    # Chesapeake Bay Retriever
    # Newfoundland
    #
    # Sex:
    # Intact Female
    #
    # Age:
    # 26 - 156 weeks
    # --------------------------------------------------------

    elif filter_type == 'water':

        query = {
            'animal_type': 'Dog',

            'breed': {
                '$in': [
                    'Labrador Retriever Mix',
                    'Chesapeake Bay Retriever',
                    'Newfoundland'
                ]
            },

            'sex_upon_outcome': 'Intact Female',

            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }


    # --------------------------------------------------------
    # Mountain / Wilderness Rescue
    #
    # Breeds:
    # German Shepherd
    # Alaskan Malamute
    # Old English Sheepdog
    # Siberian Husky
    # Rottweiler
    #
    # Sex:
    # Intact Male
    #
    # Age:
    # 26 - 156 weeks
    # --------------------------------------------------------

    elif filter_type == 'mountain':

        query = {
            'animal_type': 'Dog',

            'breed': {
                '$in': [
                    'German Shepherd',
                    'Alaskan Malamute',
                    'Old English Sheepdog',
                    'Siberian Husky',
                    'Rottweiler'
                ]
            },

            'sex_upon_outcome': 'Intact Male',

            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }


    # --------------------------------------------------------
    # Disaster / Individual Tracking
    #
    # Breeds:
    # Doberman Pinscher
    # German Shepherd
    # Golden Retriever
    # Bloodhound
    # Rottweiler
    #
    # Sex:
    # Intact Male
    #
    # Age:
    # 20 - 300 weeks
    # --------------------------------------------------------

    elif filter_type == 'disaster':

        query = {
            'animal_type': 'Dog',

            'breed': {
                '$in': [
                    'Doberman Pinscher',
                    'German Shepherd',
                    'Golden Retriever',
                    'Bloodhound',
                    'Rottweiler'
                ]
            },

            'sex_upon_outcome': 'Intact Male',

            'age_upon_outcome_in_weeks': {
                '$gte': 20,
                '$lte': 300
            }
        }


    # --------------------------------------------------------
    # Fallback
    # --------------------------------------------------------

    else:

        query = {}


    # --------------------------------------------------------
    # Query MongoDB through CRUD module
    # --------------------------------------------------------

    filtered_data = db.read(query)

    filtered_df = pd.DataFrame.from_records(filtered_data)


    # Remove MongoDB ObjectID if present.
    if '_id' in filtered_df.columns:
        filtered_df.drop(columns=['_id'], inplace=True)


    return filtered_df.to_dict('records')


# ============================================================
# Pie Chart
# ============================================================

@app.callback(
    Output('graph-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:

        return [
            dcc.Graph(
                figure=px.pie(
                    title='No animals match the selected filter'
                )
            )
        ]


    dff = pd.DataFrame.from_dict(viewData)


    # Count animals by breed.
    breed_counts = (
        dff['breed']
        .value_counts()
        .reset_index()
    )

    breed_counts.columns = ['breed', 'count']

    # Limit the unfiltered dashboard to the 10 most common breeds.
    # Filtered rescue views will normally contain only a few breeds.
    breed_counts = breed_counts.sort_values(
        'count',
        ascending=False
    )

    if len(breed_counts) > 10:
        breed_counts = breed_counts.head(10)

    breed_counts = breed_counts.sort_values(
        'count',
        ascending=True
    )

    # Create a horizontal bar chart.
    fig = px.bar(
        breed_counts,
        x='count',
        y='breed',
        orientation='h',
        title='Top 10 Breeds by Animal Count',
        labels={
            'count': 'Number of Animals',
            'breed': 'Breed'
        }
    )

    fig.update_layout(
        margin=dict(
            l=20,
            r=20,
            t=60,
            b=20
        ),
        height=500
    )

    return [
        dcc.Graph(
            figure=fig,
            style={
                'height': '500px'
            }
        )
    ]


# ============================================================
# Highlight Selected Columns
# ============================================================

@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    if not selected_columns:
        return []

    return [
        {
            'if': {
                'column_id': i
            },

            'background_color': '#D2F3FF'
        }

        for i in selected_columns
    ]


# ============================================================
# Geolocation Chart
# ============================================================

@app.callback(
    Output('map-id', 'children'),

    [
        Input('datatable-id', 'derived_virtual_data'),
        Input('datatable-id', 'derived_virtual_selected_rows')
    ]
)
def update_map(viewData, index):

    if viewData is None or len(viewData) == 0:
        return []


    dff = pd.DataFrame.from_dict(viewData)


    # If no row is selected, use the first row.
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]


    # Make sure the selected row exists.
    if row >= len(dff):
        row = 0


    # Get coordinates using column names rather than
    # hard-coded column positions.
    latitude = dff.iloc[row]['location_lat']
    longitude = dff.iloc[row]['location_long']

    breed = dff.iloc[row]['breed']
    animal_name = dff.iloc[row]['name']


    return [
        dl.Map(

            style={
                'width': '100%',
                'height': '500px'
            },

            center=[
                30.75,
                -97.48
            ],

            zoom=10,

            children=[

                dl.TileLayer(
                    id='base-layer-id'
                ),

                dl.Marker(

                    position=[
                        latitude,
                        longitude
                    ],

                    children=[

                        dl.Tooltip(
                            breed
                        ),

                        dl.Popup([

                            html.H3(
                                'Animal Name'
                            ),

                            html.P(
                                animal_name
                            ),

                            html.P(
                                'Breed: ' + str(breed)
                            )
                        ])
                    ]
                )
            ]
        )
    ]


# ============================================================
# Run Dashboard
# ============================================================

app.run_server()

Dash app running on https://capitaltempo-charmmessage-3000.codio.io/proxy/8050/
